## **1. The Core Architecture**

- Under the hood, psutil wraps native operating system APIs (like /proc on Linux, kvm on macOS, and the Windows API). It abstracts these differences into clean, pythonic objects.

- Every operation generally falls into two buckets:
    - System-wide metrics: High-level functions for hardware utilization (CPU, RAM, Disks, Network).
    - Process-specific metrics: The psutil.Process object, which points to a specific PID (Process ID) to inspect or control its lifecycle.

In [1]:
import os
import sys
import time
import psutil
import subprocess


## **2. System Monitoring (The Basics)**

### **CPU Usage**

In [2]:
# Logical vs Physical Cores
print(f"Phsical Cores : {psutil.cpu_count(logical=False)}")
print(f"Logical Cores (Threads) : {psutil.cpu_count()}")

Phsical Cores : 2
Logical Cores (Threads) : 4


> To understand the difference between physical CPUs and logical CPUs (threads), it helps to think of your computer's processor as a kitchen.

- **1. Physical CPU (The Cook)**
    - A **Physical CPU** (often called a **Core**) is the actual, physical hardware etched onto the silicon chip. It contains its own execution circuits, ALU (Arithmetic Logic Unit), and cache memory.
    * **The Analogy:** Think of a physical core as a **human chef** in a kitchen.
        * If you have a quad-core processor, you have 4 independent chefs working on your computer's tasks. They can work completely simultaneously without getting in each other's way.

- **2. What is a Thread? (The Recipe/Task)**
    - In software, a **Thread** is the smallest sequence of programmed instructions that can be managed independently by an operating system.
    * **The Analogy:** A thread is a **specific recipe or task** given to a chef (e.g., "chop the onions" or "stir the sauce").
        * A single application (like a web browser) can open multiple threads at the same time—one thread handles the user interface, another downloads a file, and a third plays audio.

- **3. Logical CPU (The Prep Station)**
    - A **Logical CPU** is a core that the operating system *perceives* as an independent processor, even if it shares physical hardware. This is made possible by a technology called **Hyper-Threading** (Intel) or **Simultaneous Multithreading / SMT** (AMD).
    * **The Analogy:** Imagine giving our chef **two separate prep stations** (two cutting boards, two sets of knives).
        * The chef can only physically chop one thing at a time. However, while the onions on Station A are simmering in a pan, the chef doesn't stand idle; they instantly turn to Station B and start prepping the carrots.
        * To the restaurant manager (the Operating System), it looks like there are two chefs working because tasks are moving twice as fast through the kitchen. In reality, it's just **one chef managing two tasks efficiently** by eliminating idle downtime.

---

- **Direct Comparison**

| Feature | Physical CPU (Core) | Logical CPU (Thread) |
| --- | --- | --- |
| **What is it?** | Real, physical hardware on the chip. | A virtual/logical execution slot recognized by the OS. |
| **Resource Sharing** | Has its own dedicated processing units. | Shares execution hardware with another logical core. |
| **Performance Impact** | Adding more cores doubles raw processing capacity. | Adding hyper-threads increases efficiency by roughly 15-30% by reducing idle time. |
| **Visibility** | If you have 8 physical cores with Hyper-Threading, Task Manager/Activity Monitor will display **16 Logical CPUs**. |  |

In [3]:
# Current utilization (blocking for 1 second to get an accurate reading)
print(f"Total CPU Usage: {psutil.cpu_percent(interval=1.0)}")
# Per-core utilization
print(f"Pre-Core CPU Usage : {psutil.cpu_percent(interval=0.1, percpu=True)}")

Total CPU Usage: 53.7
Pre-Core CPU Usage : [70.0, 60.0, 90.0, 50.0]


## **Memory RAM**

In [5]:
mem = psutil.virtual_memory()
# meme return a named tuple with bytes
print(f"Total RAM : {mem.total / (1024 ** 3):.2f} GB")
print(f"Avaliable RAM : {mem.available / (1024 ** 3):.2f} GB")
print(f"RAM Percentage Used : {mem.used / (1024 ** 3):.2f} %")

Total RAM : 23.23 GB
Avaliable RAM : 17.23 GB
RAM Percentage Used : 6.01 %


## **Disk**

In [6]:
disk_usage = psutil.disk_usage('/home/el7m7')
print(f"Disk Usage : {disk_usage.percent} %")

# Get read/write since boot
disk_io = psutil.disk_io_counters()
print(f"Read Count : {disk_io.read_count}, Write Count : {disk_io.write_count}")

Disk Usage : 28.9 %
Read Count : 145916, Write Count : 36305


## **Network**

In [7]:
# Bytes sent/received across all interface
net_io = psutil.net_io_counters()
print(f"Bytes sent : {net_io.bytes_sent}, Bytes received : {net_io.bytes_recv}")

Bytes sent : 45435167, Bytes received : 227582940


## **3. Process Management & Introspection (Intermediate)**

- To monitor or manage processes, you instantiate a psutil.Process object using its PID. If you don't pass a PID, it defaults to the current Python script's PID (os.getpid()).

- **Safely Inspecting Processes**
    - Processes are highly dynamic. A process can terminate mid-execution, throwing a psutil.NoSuchProcess error. Always write defensive code using try/except.

In [2]:
def inspect_process(pid):
    try:
        p = psutil.Process(pid)

        # Basic MetaData
        print(f"Name: {p.name()}")
        print(f"execute path : {p.exe()}")
        print(f"Status : {p.status()}")
        print(f"Parent PID : {p.ppid()}")

        # Resource used by this specific process
        print(f"CPU Percente: {p.cpu_percent(interval=0.1)}%")
        # rss = Resident Set Size (physical memory the process has allocated)
        # vms = Virtual Memory Size
        mem_info = p.memory_info()
        print(f"Physical Memory Used: {mem_info.rss / (1024**2):.2f} MB")

        # Open file and Network Connections
        print(f"Open files {p.open_files()}")
        print(f"Network Connections : {p.net_connections()}")

    except psutil.NoSuchProcess:
        print(f"Process with PID {pid} does not exists anymore.")
    except psutil.AccessDenied:
        print(f"You don't have administrative privilieges to inspect PID {pid}")

In [3]:
inspect_process(os.getpid())

Name: python
execute path : /home/el7m7/anaconda3/envs/env/bin/python3.10
Status : running
Parent PID : 7272
CPU Percente: 0.0%
Physical Memory Used: 64.29 MB
Open files [popenfile(path='/home/el7m7/.config/Code/logs/20260713T091622/window1/exthost/exthost.log', fd=45, position=204570, mode='a', flags=33793), popenfile(path='/home/el7m7/.config/Code/logs/20260713T091622/window1/exthost/extHostTelemetry.log', fd=46, position=0, mode='a', flags=33793), popenfile(path='/usr/share/code/resources/app/node_modules.asar', fd=49, position=580, mode='r', flags=32768), popenfile(path='/home/el7m7/.ipython/profile_default/history.sqlite', fd=54, position=0, mode='r+', flags=688130), popenfile(path='/home/el7m7/.ipython/profile_default/history.sqlite', fd=55, position=0, mode='r+', flags=688130), popenfile(path='/home/el7m7/.config/Code/logs/20260713T091622/window1/exthost/ms-python.python/Python.log', fd=61, position=2184, mode='a', flags=33793), popenfile(path='/home/el7m7/.config/Code/logs/2026

- **Iterating and Filtering the Process Tree**
    - If you want to find a process by name (e.g., finding all python or chrome tasks), use psutil.process_iter(). Pass the attributes you want to fetch directly into the iterator to optimize OS system calls.

In [11]:
# Efficiently search for all python processes
for process in psutil.process_iter(['pid', 'name', 'username']):
    try:
        if 'python' in process.info['name'].lower():
            print(process.info)
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass

{'username': 'el7m7', 'pid': 37967, 'name': 'python'}


## **4. Advanced Concepts & System Automation (Master Level)**

### **High-Performance Process Tracking**

- Iterating through all processes continuously is expensive. If you are monitoring a specific application spawn, track it hierarchically using children().

- Below is an automation example that starts a subprocess, tracks its resource usage dynamically, and safely kills it if it violates a memory threshold (e.g., memory leak protection).

In [13]:
def monitor_and_inforce_limites(command, max_mb=500):
    # lanch an external long-running job
    child_process = subprocess.Popen(command)
    pid = child_process.pid

    try:
        # Wrap it with psutil.Process
        p = psutil.Process(pid)

        while child_process.poll() is None: # while process is running
            #Update Memory metrics
            rss_mb = p.memory_info().rss / (1024 ** 2)
            print(f"PID {pid} Current Memory Usage : {rss_mb:.2f} MB")

            if rss_mb > max_mb:
                print(f"CRITICAL : PID {pid} exceeded {max_mb} MB limit. Terminating...")

                # Clean Termination kill children first then parent 
                for child in p.children(recursive=True):
                    child.kill()
                p.kill()
                break
            time.sleep(2)
    
    except psutil.NoSuchProcess:
        print("Process Completed before Monitor attached")

In [16]:
cmd = [sys.executable, "memory_hog.py"]

In [17]:
print("Start Monitoring test ....")
monitor_and_inforce_limites(cmd, 100)

Start Monitoring test ....
PID 57831 Current Memory Usage : 1.38 MB
Memory hog started... eating RAM now.
Allocated more memory...
Allocated more memory...
PID 57831 Current Memory Usage : 47.39 MB
Allocated more memory...
Allocated more memory...
PID 57831 Current Memory Usage : 87.39 MB
Allocated more memory...
Allocated more memory...
PID 57831 Current Memory Usage : 127.40 MB
CRITICAL : PID 57831 exceeded 100 MB limit. Terminating...


### **CPU Affinity (Performance Profiling)**

- In high-performance applications (like complex data pipelines or simulators), you might want to pin a heavy process to specific CPU cores so it doesn't interfere with the OS scheduler or cause cache misses. This is called CPU Affinity.

In [18]:
p = psutil.Process() # Current Process
print(f"Current Core affinity : {p.cpu_affinity()}")


Current Core affinity : [0, 1, 2, 3]


In [19]:
# Pin this script to Only execute on Core 0 and Core 1
p.cpu_affinity([0, 1])
print(f"Nex Core Affinity : {p.cpu_affinity()}")

Nex Core Affinity : [0, 1]


In [20]:
# to realese back to all cores
# p.cpu_affinity([])

### **Advanced Networking: Real-time Port Scanning**

- You can use psutil.net_connections() to check what ports are open on your machine, what PIDs are listening on them, and what remote IPs they are talking to.

In [21]:
# Get all active TCP connections that are currently LISTENING for incoming traffic
connections = psutil.net_connections(kind = "tcp")

for conn in connections:
    if conn.status == psutil.CONN_LISTEN:
        print(f"Interface IP: {conn.laddr.ip} | Port: {conn.laddr.port} | Handled by PID: {conn.pid}")

Interface IP: :: | Port: 54113 | Handled by PID: 4513
Interface IP: 127.0.0.1 | Port: 9009 | Handled by PID: 37967
Interface IP: 127.0.0.1 | Port: 9006 | Handled by PID: 37967
Interface IP: 127.0.0.1 | Port: 9007 | Handled by PID: 37967
Interface IP: 127.0.0.1 | Port: 8828 | Handled by PID: 4513
Interface IP: 127.0.0.1 | Port: 45441 | Handled by PID: 37967
Interface IP: 127.0.0.1 | Port: 27182 | Handled by PID: 2011
Interface IP: 127.0.0.1 | Port: 9008 | Handled by PID: 37967
Interface IP: 127.0.0.1 | Port: 33921 | Handled by PID: 2011
Interface IP: 0.0.0.0 | Port: 54112 | Handled by PID: 4843
Interface IP: 127.0.0.1 | Port: 9005 | Handled by PID: 37967
Interface IP: 127.0.0.1 | Port: 2680 | Handled by PID: 2011


## **Architecture Checklist for Production Monitoring**

1. Reuse Process Instances: Don't call psutil.Process(pid) every loop iteration. Instantiate it once and call methods like p.memory_info() on that same object; it caches static properties like name and path to maximize speed.

2. Zombie Cleanup: If you spawn background processes on Linux, they can become "zombies" when they die if the parent doesn't read their exit status. Use psutil.wait_procs() to clean up pools of child processes gracefully.